In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!git clone https://github.com/laureneproctor/Olympiad-AI.git
%cd Olympiad-AI

Cloning into 'Olympiad-AI'...
remote: Enumerating objects: 1108, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 1108 (delta 110), reused 16 (delta 14), pack-reused 935 (from 2)
Receiving objects: 100% (1108/1108), 1.32 MiB | 10.18 MiB/s, done.
Resolving deltas: 100% (648/648), done.
/content/Olympiad-AI/Olympiad-AI


In [12]:
import os
import json
import yaml
import importlib
from pathlib import Path

In [13]:
import aimo3.scripts.helpers as helper
import aimo3.scripts.evaluate as evaluate
importlib.reload(helper)
importlib.reload(evaluate)

from aimo3.scripts.helpers import load_yaml

In [14]:
# --- user choices ---
MODEL_KEY = "deepseekmath"      # or "qwen"
EXPERIMENT_NAME = "exp2"        # should match an experiment in data.yaml
SPLIT_NAME = "test"             # "train", "val", or "test"

# Keep these aligned with your Drive structure
PREPARED_SPLITS_ROOT = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3"
OUTPUT_ROOT = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/runs"

# Evaluation caps
PASS1_MAX_ITEMS = 50
MAJN_MAX_ITEMS = 50
MAJN_N = 8

# Source configs from repo
DATA_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/data.yaml"
SFT_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/sft.yaml"
BASE_EVAL_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/evaluate.yaml"

# Output config for this notebook
BASELINE_EVAL_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/eval_baseline.yaml"

In [15]:
data_cfg = load_yaml(DATA_CONFIG_PATH)
sft_cfg = load_yaml(SFT_CONFIG_PATH)
base_eval_cfg = load_yaml(BASE_EVAL_CONFIG_PATH)

assert MODEL_KEY in data_cfg["models"], f"Unknown MODEL_KEY: {MODEL_KEY}"
assert EXPERIMENT_NAME in data_cfg["experiments"], f"Unknown EXPERIMENT_NAME: {EXPERIMENT_NAME}"

base_model_name = data_cfg["models"][MODEL_KEY]["name"]
n_rows = data_cfg["experiments"][EXPERIMENT_NAME]["N"]
dataset_path = os.path.join(PREPARED_SPLITS_ROOT, "splits", str(n_rows))

print("Base model:", base_model_name)
print("Dataset path:", dataset_path)

Base model: deepseek-ai/deepseek-math-7b-instruct
Dataset path: /content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000


In [16]:
# Update the in-memory SFT config so metadata matches the baseline run we want
sft_cfg["run"]["experiment_name"] = EXPERIMENT_NAME
sft_cfg["run"]["model_key"] = MODEL_KEY
sft_cfg["paths"]["prepared_splits_root"] = PREPARED_SPLITS_ROOT
sft_cfg["paths"]["output_root"] = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models"

# Save a temporary SFT config that evaluate.py can read
BASELINE_SFT_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/sft_baseline_temp.yaml"
with open(BASELINE_SFT_CONFIG_PATH, "w") as f:
    yaml.safe_dump(sft_cfg, f, sort_keys=False)

baseline_eval_cfg = base_eval_cfg.copy()
baseline_eval_cfg["paths"] = baseline_eval_cfg.get("paths", {})
baseline_eval_cfg["paths"]["data_config_path"] = DATA_CONFIG_PATH
baseline_eval_cfg["paths"]["sft_config_path"] = BASELINE_SFT_CONFIG_PATH
baseline_eval_cfg["paths"]["model_checkpoint"] = base_model_name
baseline_eval_cfg["paths"]["dataset_path"] = dataset_path
baseline_eval_cfg["paths"]["output_dir"] = os.path.join(
    OUTPUT_ROOT,
    f"evaluation_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}"
)

baseline_eval_cfg["evaluation"] = baseline_eval_cfg.get("evaluation", {})
baseline_eval_cfg["evaluation"]["split"] = SPLIT_NAME
baseline_eval_cfg["evaluation"]["max_items"] = PASS1_MAX_ITEMS
baseline_eval_cfg["evaluation"]["majn_max_items"] = MAJN_MAX_ITEMS
baseline_eval_cfg["evaluation"]["majn_n_samples"] = MAJN_N

baseline_eval_cfg["reporting"] = baseline_eval_cfg.get("reporting", {})
baseline_eval_cfg["reporting"]["save_report"] = True
baseline_eval_cfg["reporting"]["report_filename"] = f"evaluation_report_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.json"

with open(BASELINE_EVAL_CONFIG_PATH, "w") as f:
    yaml.safe_dump(baseline_eval_cfg, f, sort_keys=False)

print("Wrote:", BASELINE_EVAL_CONFIG_PATH)
print(json.dumps(baseline_eval_cfg, indent=2))

Wrote: /content/Olympiad-AI/aimo3/configs/eval_baseline.yaml
{
  "run": {
    "seed": 42
  },
  "paths": {
    "data_config_path": "/content/Olympiad-AI/aimo3/configs/data.yaml",
    "sft_config_path": "/content/Olympiad-AI/aimo3/configs/sft_baseline_temp.yaml",
    "model_checkpoint": "deepseek-ai/deepseek-math-7b-instruct",
    "dataset_path": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000",
    "output_dir": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/runs/evaluation_baseline_deepseekmath_exp2"
  },
  "evaluation": {
    "split": "test",
    "max_items": 50,
    "eval_pass1": true,
    "pass1_max_new_tokens": 512,
    "eval_majn": true,
    "majn_n_samples": 8,
    "majn_max_items": 50,
    "majn_max_new_tokens": 512,
    "majn_temperature": 0.7,
    "majn_top_p": 0.95
  },
  "reporting": {
    "save_report": true,
    "report_filename": "evaluation_report_baseline_deepseekmath_exp2.json",
    "verbose

In [17]:
import importlib
import aimo3.scripts.evaluate as evaluate
importlib.reload(evaluate)

baseline_results = evaluate.run_evaluation(BASELINE_EVAL_CONFIG_PATH)

Experiment: exp2
Model key: deepseekmath
Model name: deepseek-ai/deepseek-math-7b-instruct
Dataset path: /content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000
Checkpoint: deepseek-ai/deepseek-math-7b-instruct
Eval split: test
Loading evaluation dataset from: /content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000
Loaded eval rows: 10035
Columns: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode', 'used_in_kaggle']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Running pass@1 evaluation...


Pass@1 Evaluation: 100%|██████████| 50/50 [03:29<00:00,  4.19s/example]


Running maj@N evaluation...


Maj@8 Evaluation: 100%|██████████| 50/50 [05:55<00:00,  7.12s/example]



=== Preview Examples ===

--- Example 1/3 ---
Gold: 432
Pass@1 pred: 2 | correct: False
Maj@N pred: 2 | correct: False
Agreement: 0.375 | vote_counts: {'0': 2, '2': 3, '162': 1, '18': 1, '90': 1}
Problem preview: Complex numbers $a,$ $b,$ $c$ form an equilateral triangle with side length 18 in the complex plane.  If $|a + b + c| = 36,$ find $|ab + ac + bc|.$
Pass@1 text preview: Let $a = \omega$ and $b = 1.$  Then $c = \omega^2,$ and \begin{align*} ab + ac + bc &= \omega + \omega^2 + 1 + \omega^2 + \omega + \omega^2 \\ &= \omega + \omega^2 + 1 + \omega^2 +
Sampled generations (extracted answer | matches gold):
  [1] ans=0 | match=False | text=Let $A,$ $B,$ and $C$ be the complex numbers corresponding to $a,$ $b,$ and $c,$ respectively.  Without loss of generality, assume that $A = 0,$ $B = 18,$ and $C = 18 \omega,$ wher
  [2] ans=2 | match=False | text=Since $a,$ $b,$ $c$ form an equilateral triangle with side length 18, we can write $b = a + 18\omega,$ $c = a + 18,$ where $\omega$ is

In [18]:
print(json.dumps(baseline_results, indent=2))

{
  "metadata": {
    "experiment_name": "exp2",
    "model_key": "deepseekmath",
    "model_name": "deepseek-ai/deepseek-math-7b-instruct",
    "checkpoint": "deepseek-ai/deepseek-math-7b-instruct",
    "dataset_path": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000",
    "split": "test",
    "n_dataset_rows": 10035
  },
  "pass1": {
    "pass@1": 0.32,
    "format_valid_rate": 0.84,
    "n": 50
  },
  "majn": {
    "maj@8": 0.22,
    "per_sample_format_valid_rate": 0.9275,
    "per_problem_any_valid_rate": 1.0,
    "avg_agreement_rate": 0.4761666666666667,
    "n": 50
  }
}
